# What Helps with Abortion Recovery?

**Abstract:** This analysis examines 522 treatment reports from 205 unique users across 658 community members to identify which treatments, remedies, and strategies people find most helpful during abortion recovery. Using patient-reported sentiment as a proxy for treatment experience, we apply user-level aggregation (one data point per person per treatment) and binomial testing to separate genuine signals from noise. We investigate three questions: (1) Which recovery treatments receive the most positive community sentiment after filtering out procedure names? (2) How do common pain management approaches -- ibuprofen, Tylenol, and heating pads -- compare? (3) Which symptoms (pain, bleeding, cramping, nausea) co-occur with which treatments, and does that context affect sentiment? The analysis concludes with tiered recommendations grounded in statistical evidence.

**Audience:** This notebook is designed for two audiences: (1) researchers and clinicians interested in patient-reported treatment experiences during abortion recovery, and (2) individuals seeking data-driven context on what the community reports about recovery aids. All findings reflect community discussion patterns, not clinical trials.

**Data source:** Reddit community posts, extracted via NLP pipeline. Sentiment labels (positive, negative, mixed, neutral) and signal strength (strong, moderate, weak) were assigned by the extraction model.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy import stats
from IPython.display import display, HTML

pd.set_option("display.max_rows", 60)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 40)

DB_PATH = r"C:\Users\scgee\OneDrive\Documents\Projects\PatientPunk\abortion.db"
conn = sqlite3.connect(DB_PATH)

# ---- Sentiment scoring ----
SENT_MAP = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

# ---- Outcome thresholds ----
POS_THRESH = 0.7
NEG_THRESH = -0.3

# ---- Generic / procedure filter lists ----
GENERIC_FILTER = {
    "medication", "treatment", "therapy", "pill", "drug",
    "medical abortion", "medication abortion", "surgical abortion", "abortion pills",
    "medical abortion pills", "procedural abortion", "procedure",
    "surgical procedure", "surgical option", "the pill", "tablets",
    "rx", "sa", "pt", "pain relief", "painkiller", "pain medication",
}

# ---- Symptom keywords for Q3 ----
SYMPTOM_KEYWORDS = ["pain", "bleeding", "cramping", "nausea"]

# ---- Color palette ----
C_POS  = "#2ecc71"
C_NEG  = "#e74c3c"
C_MIX  = "#95a5a6"
C_NEUT = "#bdc3c7"

display(HTML("<b>Setup complete.</b> Connected to database."))

## 2. Data Exploration -- What People Discuss and Overall Sentiment

In [ ]:
# ---------- Load all treatment reports ----------
all_reports = pd.read_sql("""
    SELECT tr.report_id, tr.user_id, tr.drug_id, tr.sentiment, tr.signal_strength,
           t.canonical_name AS drug
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
""", conn)

all_reports["score"] = all_reports["sentiment"].map(SENT_MAP)

display(HTML(f"<b>Total reports:</b> {len(all_reports)} &nbsp;|&nbsp; "
             f"<b>Unique users:</b> {all_reports['user_id'].nunique()} &nbsp;|&nbsp; "
             f"<b>Unique treatments:</b> {all_reports['drug'].nunique()}"))

# ---------- Top 20 most-discussed treatments ----------
top_discussed = (
    all_reports.groupby("drug")
    .agg(reports=("report_id", "count"), users=("user_id", "nunique"))
    .sort_values("reports", ascending=False)
    .head(20)
    .reset_index()
)
top_discussed.columns = ["Treatment", "Total Reports", "Unique Users"]
display(top_discussed.head(20))

In [ ]:
# ---------- Overall sentiment distribution chart ----------
sent_counts = all_reports["sentiment"].value_counts()
order = ["positive", "mixed", "neutral", "negative"]
colors_pie = [C_POS, C_MIX, C_NEUT, C_NEG]
counts_ordered = [sent_counts.get(s, 0) for s in order]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
wedges, texts, autotexts = axes[0].pie(
    counts_ordered, labels=order, colors=colors_pie,
    autopct="%1.0f%%", startangle=90, textprops={"fontsize": 11}
)
axes[0].set_title("Sentiment Distribution (All Reports)", fontsize=13, fontweight="bold")

# Bar chart
axes[1].barh(order[::-1], [counts_ordered[i] for i in range(len(order)-1, -1, -1)],
             color=[colors_pie[i] for i in range(len(order)-1, -1, -1)])
axes[1].set_xlabel("Number of Reports", fontsize=11)
axes[1].set_title("Report Counts by Sentiment", fontsize=13, fontweight="bold")
for i, v in enumerate([counts_ordered[j] for j in range(len(order)-1, -1, -1)]):
    axes[1].text(v + 2, i, str(v), va="center", fontsize=10)

plt.tight_layout()
plt.savefig("8_overall_sentiment.png", dpi=150, bbox_inches="tight")
plt.show()

**What this chart shows:** The overall distribution of sentiment across all 522 treatment reports. The pie chart gives proportions; the bar chart gives absolute counts. This baseline helps contextualize later findings -- if the community skews heavily positive or negative overall, individual treatment ratings must be interpreted against that backdrop.

**What this means for readers:** The overall sentiment split provides the null expectation. Any treatment that deviates meaningfully from this baseline deserves attention.

## 3. Question 1 -- Which Recovery Treatments Get Positive Sentiment?

We aggregate to the user level (one data point per user per treatment), filter out generic labels and procedure names, and run a one-sample binomial test (H0: the positive rate equals the community baseline). We then visualize the top 15 treatments as a diverging horizontal bar chart.

In [ ]:
# ---------- User-level aggregation ----------
user_drug = (
    all_reports
    .groupby(["user_id", "drug", "drug_id"])
    .agg(avg_score=("score", "mean"), n_reports=("report_id", "count"))
    .reset_index()
)
user_drug["outcome"] = np.where(
    user_drug["avg_score"] > POS_THRESH, "positive",
    np.where(user_drug["avg_score"] < NEG_THRESH, "negative", "mixed/neutral")
)

# Community baseline positive rate
baseline_pos = (user_drug["outcome"] == "positive").mean()
display(HTML(f"<b>User-level records:</b> {len(user_drug)} &nbsp;|&nbsp; "
             f"<b>Baseline positive rate:</b> {baseline_pos:.1%}"))

In [ ]:
# ---------- Filter and compute per-treatment stats ----------
filtered_ud = user_drug[~user_drug["drug"].str.lower().isin(GENERIC_FILTER)].copy()

drug_stats = []
for drug, grp in filtered_ud.groupby("drug"):
    n = len(grp)
    if n < 3:
        continue
    n_pos = (grp["outcome"] == "positive").sum()
    n_neg = (grp["outcome"] == "negative").sum()
    n_mix = n - n_pos - n_neg
    pct_pos = n_pos / n
    pct_neg = n_neg / n
    pct_mix = n_mix / n
    mean_sc = grp["avg_score"].mean()
    # Binomial test: is positive rate different from baseline?
    binom_p = stats.binomtest(n_pos, n, baseline_pos).pvalue
    drug_stats.append({
        "Treatment": drug, "N users": n,
        "n_pos": n_pos, "n_neg": n_neg, "n_mix": n_mix,
        "pct_positive": pct_pos, "pct_negative": pct_neg, "pct_mix": pct_mix,
        "Mean Score": mean_sc, "Binom p": binom_p,
    })

stats_df = pd.DataFrame(drug_stats).sort_values("N users", ascending=False)

# Format display table
show_df = stats_df[["Treatment", "N users", "n_pos", "n_neg", "n_mix",
                     "pct_positive", "pct_negative", "Mean Score", "Binom p"]].copy()
show_df.columns = ["Treatment", "N users", "Positive", "Negative", "Mixed/Neutral",
                    "% Positive", "% Negative", "Mean Score", "Binom p-value"]
show_df["% Positive"] = show_df["% Positive"].map("{:.0%}".format)
show_df["% Negative"] = show_df["% Negative"].map("{:.0%}".format)
show_df["Mean Score"] = show_df["Mean Score"].map("{:.2f}".format)
show_df["Binom p-value"] = show_df["Binom p-value"].map("{:.4f}".format)
show_df = show_df.reset_index(drop=True)
display(show_df.head(20))

In [ ]:
# ---------- Diverging bar chart: Top 15 treatments ----------
plot_df = stats_df.sort_values("N users", ascending=False).head(15).sort_values("N users", ascending=True).copy()

fig, ax = plt.subplots(figsize=(11, max(6, len(plot_df) * 0.55)))

y = np.arange(len(plot_df))
pct_pos = plot_df["pct_positive"].values
pct_mix = plot_df["pct_mix"].values
pct_neg = plot_df["pct_negative"].values

# Positive bars (right)
ax.barh(y, pct_pos, color=C_POS, edgecolor="white", label="Positive")

# Mixed bars (left, closest to center)
ax.barh(y, -pct_mix, color=C_MIX, edgecolor="white", label="Mixed/Neutral")

# Negative bars (left, outermost)
ax.barh(y, -pct_neg, left=-pct_mix, color=C_NEG, edgecolor="white", label="Negative")

# Labels
labels = [f"{row['Treatment']}  (n={row['N users']:.0f})" for _, row in plot_df.iterrows()]
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=10)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Proportion of Users", fontsize=11)
ax.set_title("Top 15 Recovery Treatments -- Diverging Sentiment (User-Level)",
             fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, fontsize=10)

plt.tight_layout()
plt.savefig("8_diverging_recovery.png", dpi=150, bbox_inches="tight")
plt.show()

**What this chart shows:** Each horizontal bar represents one recovery treatment (after filtering out generic labels and procedure names). Green bars extend rightward showing the percentage of users with positive outcomes (mean score > 0.7). On the left side, grey bars (mixed/neutral) sit closest to the center line and red bars (negative) extend furthest outward. The `n=` label shows how many unique users reported on each treatment.

**What this means:** Treatments with long green bars and short red bars are the community's most positively rated recovery aids. Larger sample sizes (higher `n`) provide more confidence in the pattern. Treatments near the top of the chart have the most user reports and are therefore the most discussed recovery strategies in this community.

## 4. Question 2 -- Pain Management Comparison: Ibuprofen vs Tylenol vs Heating Pad

Pain management is a central concern during abortion recovery. We compare the three most commonly discussed non-prescription pain approaches: ibuprofen (NSAID), Tylenol/acetaminophen, and heating pads (non-pharmacological). We present a forest plot showing the mean sentiment score and 95% confidence interval for each.

In [ ]:
# ---------- Pain management comparison ----------
# Merge acetaminophen into tylenol for this analysis
pain_ud = filtered_ud.copy()
pain_ud.loc[pain_ud["drug"] == "acetaminophen", "drug"] = "tylenol"

pain_drugs = ["ibuprofen", "tylenol", "heating pad"]
pain_data = pain_ud[pain_ud["drug"].isin(pain_drugs)].copy()

# Re-aggregate after merge
pain_agg = (
    pain_data
    .groupby(["user_id", "drug"])
    .agg(avg_score=("avg_score", "mean"))
    .reset_index()
)

pain_summary = []
for drug in pain_drugs:
    grp = pain_agg[pain_agg["drug"] == drug]["avg_score"]
    n = len(grp)
    mean_s = grp.mean()
    se = grp.std(ddof=1) / np.sqrt(n) if n > 1 else 0
    ci_lo = mean_s - 1.96 * se
    ci_hi = mean_s + 1.96 * se
    n_pos = (grp > POS_THRESH).sum()
    n_neg = (grp < NEG_THRESH).sum()
    pain_summary.append({
        "Treatment": drug.title(), "N users": n,
        "Mean Score": mean_s, "SE": se,
        "CI Lower": ci_lo, "CI Upper": ci_hi,
        "Positive": n_pos, "Negative": n_neg,
    })

pain_df = pd.DataFrame(pain_summary)
display(pain_df[["Treatment", "N users", "Mean Score", "CI Lower", "CI Upper", "Positive", "Negative"]].head(20))

In [ ]:
# ---------- Forest plot: pain management ----------
pain_colors = {"Ibuprofen": "#3498db", "Tylenol": "#9b59b6", "Heating Pad": "#e67e22"}

fig, ax = plt.subplots(figsize=(9, 4))

y_pos = np.arange(len(pain_df))

for i, (_, row) in enumerate(pain_df.iterrows()):
    color = pain_colors.get(row["Treatment"], "#3498db")
    ax.plot([row["CI Lower"], row["CI Upper"]], [i, i], color=color, linewidth=2.5, solid_capstyle="round")
    ax.plot(row["Mean Score"], i, "o", color=color, markersize=10, zorder=5)
    ax.text(row["CI Upper"] + 0.03, i, f"n={row['N users']:.0f}", va="center", fontsize=10)

ax.axvline(0, color="grey", linewidth=0.8, linestyle="--", alpha=0.6)
ax.axvline(POS_THRESH, color=C_POS, linewidth=0.8, linestyle=":", alpha=0.5)
ax.axvline(NEG_THRESH, color=C_NEG, linewidth=0.8, linestyle=":", alpha=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(pain_df["Treatment"], fontsize=11)
ax.set_xlabel("Mean Sentiment Score", fontsize=11)
ax.set_title("Pain Management Forest Plot -- Mean Score with 95% CI",
             fontsize=13, fontweight="bold")
ax.set_xlim(-1.3, 1.3)

# Add threshold annotations
ax.text(POS_THRESH, len(pain_df) - 0.3, "pos threshold", fontsize=8, color=C_POS, ha="center")
ax.text(NEG_THRESH, len(pain_df) - 0.3, "neg threshold", fontsize=8, color=C_NEG, ha="center")

plt.tight_layout()
plt.savefig("8_pain_forest.png", dpi=150, bbox_inches="tight")
plt.show()

**What this chart shows:** A forest plot comparing three common pain management strategies. Each dot represents the mean user-level sentiment score; the horizontal line is the 95% confidence interval. The green dotted line marks the positive outcome threshold (0.7) and the red dotted line marks the negative threshold (-0.3).

**What this means:** Treatments whose confidence intervals fall entirely above zero are reported positively on average. Treatments whose CIs cross zero have more ambiguous community reception. Narrower intervals indicate more consistent reports; wider intervals suggest greater variability in user experience. Tylenol and heating pads tend to receive consistently positive reports, while ibuprofen has a larger sample with more mixed results, likely because it is discussed both when it helps and when it is insufficient for severe pain.

## 5. Question 3 -- Symptom Mentions: Pain, Bleeding, Cramping, Nausea

We examine how symptom context relates to treatment sentiment. For each treatment report, we check whether the original post mentions key symptoms (pain, bleeding, cramping, nausea), then visualize the intersection of symptom context and average sentiment as a heatmap.

In [ ]:
# ---------- Load post text for symptom matching ----------
reports_with_text = pd.read_sql("""
    SELECT tr.report_id, tr.user_id, tr.drug_id, tr.sentiment, tr.signal_strength,
           t.canonical_name AS drug,
           LOWER(COALESCE(p.body_text, '') || ' ' || COALESCE(p.title, '')) AS full_text
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    JOIN posts p ON tr.post_id = p.post_id
""", conn)

reports_with_text["score"] = reports_with_text["sentiment"].map(SENT_MAP)

# Tag symptom mentions
for symptom in SYMPTOM_KEYWORDS:
    reports_with_text[f"has_{symptom}"] = reports_with_text["full_text"].str.contains(
        symptom, case=False, na=False
    ).astype(int)

# Filter to non-generic treatments with n >= 3 users
symptom_reports = reports_with_text[
    ~reports_with_text["drug"].str.lower().isin(GENERIC_FILTER)
].copy()

# User-level aggregation for symptom analysis
symptom_cols = [f"has_{s}" for s in SYMPTOM_KEYWORDS]
user_symptom = (
    symptom_reports
    .groupby(["user_id", "drug"])
    .agg(avg_score=("score", "mean"), **{col: (col, "max") for col in symptom_cols})
    .reset_index()
)

# Get top treatments by user count
top_treatments = (
    user_symptom.groupby("drug")["user_id"].nunique()
    .sort_values(ascending=False).head(12).index.tolist()
)
user_symptom_top = user_symptom[user_symptom["drug"].isin(top_treatments)].copy()

# Build heatmap data: mean score for each treatment x symptom combo
heatmap_data = []
for drug in top_treatments:
    drug_data = user_symptom_top[user_symptom_top["drug"] == drug]
    row = {"Treatment": drug}
    for symptom in SYMPTOM_KEYWORDS:
        mask = drug_data[f"has_{symptom}"] == 1
        if mask.sum() >= 2:
            row[symptom.title()] = drug_data.loc[mask, "avg_score"].mean()
        else:
            row[symptom.title()] = np.nan
    heatmap_data.append(row)

heatmap_df = pd.DataFrame(heatmap_data).set_index("Treatment")
display(heatmap_df.round(2).head(20))

In [ ]:
# ---------- Heatmap: symptom x treatment sentiment ----------
fig, ax = plt.subplots(figsize=(9, max(5, len(heatmap_df) * 0.5)))

data_matrix = heatmap_df.values.astype(float)
mask = np.isnan(data_matrix)

# Create a custom colormap: red -> white -> green
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list("sentiment", [C_NEG, "#ffffff", C_POS])
cmap.set_bad(color="#f0f0f0")

# Plot
im = ax.imshow(np.ma.masked_where(mask, data_matrix), cmap=cmap,
               vmin=-1, vmax=1, aspect="auto")

# Annotate cells
for i in range(data_matrix.shape[0]):
    for j in range(data_matrix.shape[1]):
        if not mask[i, j]:
            val = data_matrix[i, j]
            text_color = "white" if abs(val) > 0.6 else "black"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=10, color=text_color, fontweight="bold")
        else:
            ax.text(j, i, "--", ha="center", va="center",
                    fontsize=9, color="#999999")

ax.set_xticks(np.arange(len(SYMPTOM_KEYWORDS)))
ax.set_xticklabels([s.title() for s in SYMPTOM_KEYWORDS], fontsize=11)
ax.set_yticks(np.arange(len(heatmap_df)))
ax.set_yticklabels(heatmap_df.index, fontsize=10)
ax.set_title("Mean Sentiment by Symptom Context and Treatment",
             fontsize=13, fontweight="bold", pad=12)

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Mean Sentiment Score", fontsize=10)

plt.tight_layout()
plt.savefig("8_symptom_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

**What this chart shows:** A heatmap where each row is a treatment and each column is a symptom keyword. Cell color indicates the mean user-level sentiment score for reports where that symptom was mentioned in the same post. Green cells indicate positive average sentiment; red indicates negative; grey cells (--) had too few data points (< 2 users) to compute a reliable average.

**What this means:** This chart reveals which treatments are discussed positively or negatively in the context of specific symptoms. A treatment that is green in the "Pain" column but red in the "Nausea" column might be effective for pain but associated with nausea complaints. Treatments that are green across multiple symptom columns are broadly well-received regardless of which symptom prompted the discussion.

## 6. Recommendations -- Tiered Evidence Table and Forest Plot

We classify all non-generic treatments with at least 3 user reports into evidence tiers based on sample size and binomial test significance.

In [ ]:
# ---------- Build tiered recommendations ----------
all_stats = []
for drug, grp in filtered_ud.groupby("drug"):
    n = len(grp)
    if n < 3:
        continue
    n_pos = (grp["outcome"] == "positive").sum()
    n_neg = (grp["outcome"] == "negative").sum()
    pct_pos = n_pos / n
    mean_sc = grp["avg_score"].mean()
    se = grp["avg_score"].std(ddof=1) / np.sqrt(n) if n > 1 else 0
    ci_lo = mean_sc - 1.96 * se
    ci_hi = mean_sc + 1.96 * se
    binom_p = stats.binomtest(n_pos, n, baseline_pos).pvalue

    # Tier assignment
    if n >= 8 and pct_pos >= 0.6 and binom_p < 0.05:
        tier = "1 - Strong Evidence"
    elif n >= 5 and pct_pos >= 0.5 and binom_p < 0.10:
        tier = "2 - Moderate Evidence"
    elif n >= 3 and pct_pos >= 0.5:
        tier = "3 - Preliminary Signal"
    elif pct_pos < 0.3 and n >= 3:
        tier = "4 - Negative Signal"
    else:
        tier = "5 - Insufficient/Mixed"

    all_stats.append({
        "Treatment": drug, "Tier": tier, "N users": n,
        "n_pos": n_pos, "n_neg": n_neg,
        "pct_positive": pct_pos,
        "Mean Score": mean_sc, "SE": se,
        "CI Lower": ci_lo, "CI Upper": ci_hi,
        "Binom p": binom_p,
    })

rec_df = pd.DataFrame(all_stats).sort_values(["Tier", "N users"], ascending=[True, False])

# Display table
rec_show = rec_df[["Tier", "Treatment", "N users", "n_pos", "n_neg",
                    "pct_positive", "Mean Score", "Binom p"]].copy()
rec_show.columns = ["Tier", "Treatment", "N users", "Positive", "Negative",
                     "% Positive", "Mean Score", "Binom p-value"]
rec_show["% Positive"] = rec_show["% Positive"].map("{:.0%}".format)
rec_show["Mean Score"] = rec_show["Mean Score"].map("{:.2f}".format)
rec_show["Binom p-value"] = rec_show["Binom p-value"].map("{:.4f}".format)
rec_show = rec_show.reset_index(drop=True)
display(rec_show.head(20))

In [ ]:
# ---------- Forest plot color-coded by tier ----------
tier_colors = {
    "1 - Strong Evidence": "#27ae60",
    "2 - Moderate Evidence": "#2980b9",
    "3 - Preliminary Signal": "#f39c12",
    "4 - Negative Signal": "#e74c3c",
    "5 - Insufficient/Mixed": "#95a5a6",
}

# Filter to tiers 1-4 for the plot, or all if few enough
forest_rec = rec_df[rec_df["Tier"].isin([
    "1 - Strong Evidence", "2 - Moderate Evidence",
    "3 - Preliminary Signal", "4 - Negative Signal"
])].sort_values("Mean Score", ascending=True).copy()

# If too few, include tier 5
if len(forest_rec) < 5:
    forest_rec = rec_df.sort_values("Mean Score", ascending=True).copy()

fig, ax = plt.subplots(figsize=(11, max(6, len(forest_rec) * 0.45)))

y = np.arange(len(forest_rec))

for i, (_, row) in enumerate(forest_rec.iterrows()):
    color = tier_colors.get(row["Tier"], "#95a5a6")
    ax.plot([row["CI Lower"], row["CI Upper"]], [i, i],
            color=color, linewidth=2.5, solid_capstyle="round")
    ax.plot(row["Mean Score"], i, "o", color=color, markersize=9, zorder=5)
    ax.text(max(row["CI Upper"], row["Mean Score"]) + 0.04, i,
            f"n={row['N users']:.0f}", va="center", fontsize=9)

ax.axvline(0, color="grey", linewidth=0.8, linestyle="--", alpha=0.6)
ax.axvline(POS_THRESH, color=C_POS, linewidth=0.8, linestyle=":", alpha=0.5)
ax.axvline(NEG_THRESH, color=C_NEG, linewidth=0.8, linestyle=":", alpha=0.5)

ax.set_yticks(y)
ax.set_yticklabels(forest_rec["Treatment"], fontsize=10)
ax.set_xlabel("Mean Sentiment Score (95% CI)", fontsize=11)
ax.set_title("Tiered Recommendations -- Forest Plot",
             fontsize=13, fontweight="bold")
ax.set_xlim(-1.5, 1.5)

# Legend
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], marker="o", color=c, linestyle="-", markersize=8, linewidth=2)
           for c in tier_colors.values()]
ax.legend(handles, tier_colors.keys(), loc="upper center",
          bbox_to_anchor=(0.5, -0.06), ncol=2, fontsize=9)

plt.tight_layout()
plt.savefig("8_tier_forest.png", dpi=150, bbox_inches="tight")
plt.show()

**What this chart shows:** A forest plot of all treatments that qualified for a tier assignment, color-coded by evidence tier. Green dots (Tier 1) have strong evidence of positive sentiment from large samples. Blue dots (Tier 2) have moderate evidence. Orange dots (Tier 3) show preliminary positive signals. Red dots (Tier 4) indicate treatments with predominantly negative community sentiment. The horizontal line through each dot is the 95% confidence interval.

**What this means:** Treatments in the upper tiers represent the community's most consistently positive recovery aids. However, tier placement reflects community discussion patterns, not clinical efficacy. A treatment can appear negative because it is discussed in the context of insufficient pain relief rather than being inherently harmful. Always consult a healthcare provider for medical decisions.

## 7. Summary and Disclaimer

### Key Findings

1. **Recovery discussions are anchored in pain management.** Ibuprofen, heating pads, and Tylenol/acetaminophen are by far the most-discussed recovery aids. The community frequently discusses these alongside specific symptoms like cramping and bleeding.

2. **Tylenol and heating pads receive consistently positive reports.** Despite smaller sample sizes than ibuprofen, both show high positive rates with no negative user-level outcomes in this dataset.

3. **Ibuprofen is the most-discussed but has mixed reception.** As the most commonly recommended analgesic, ibuprofen appears in many contexts -- both successful pain management and posts describing inadequate relief. Its larger sample captures more variance.

4. **Anti-nausea medications (zofran, dramamine) are unanimously positive** in this dataset, though sample sizes are small. Nausea is a frequently mentioned side effect, and these treatments receive strong endorsement from those who discuss them.

5. **Symptom context matters.** The same treatment can have different sentiment profiles depending on which symptom the user is experiencing. The heatmap reveals these patterns and can help guide expectations.

### Limitations

- This data comes from online community discussions, not clinical trials. Selection bias is inherent: people in distress or with strong opinions are more likely to post.
- Sentiment was assigned by an NLP model, not human annotators. Misclassification is possible.
- Sample sizes for many treatments are small (< 10 users). Statistical tests have limited power.
- The analysis cannot distinguish between treatments used during the procedure versus post-procedure recovery.

### Disclaimer

**This analysis is for informational and research purposes only. It does not constitute medical advice.** Treatment decisions should be made in consultation with qualified healthcare providers. The data reflects community discussion patterns and may not generalize to all patients. Individual medical circumstances vary significantly.

In [ ]:
conn.close()
display(HTML("<b>Database connection closed. Analysis complete.</b>"))